# Exploring `soc` and `soccfg` Parameters

This notebook connects to the QICK hardware and queries the various properties
and methods available on the `soc` (Pyro4 proxy) and `soccfg` (QickConfig) objects.

## Setup & Connection

In [ ]:
cfg_file = 'sample_50_rfboard.yml'
ip = '10.108.30.23'
rfsoc_alias = 'bf1_soc'

In [ ]:
import os
import numpy as np
from qick import QickConfig

from slab_qick_calib.exp_handling.instrumentmanager import InstrumentManager
from slab_qick_calib.helpers import config

%load_ext autoreload
%autoreload 2

In [ ]:
cfg_path = os.path.join(os.getcwd(), '..', 'configs', cfg_file)
auto_cfg = config.load(cfg_path)

im = InstrumentManager(ns_address=auto_cfg['aliases']['ip'])
print(im)

soc = im[auto_cfg['aliases']['soc']]
soccfg = QickConfig(soc.get_cfg())

---
# 1. `soccfg` Overview

Print the built-in description which summarises generators, readouts, tProc, clocks, etc.

In [ ]:
print(soccfg)

## 1.1 Raw configuration dictionary keys

In [ ]:
raw_cfg = soccfg.get_cfg()
print('Top-level keys:', list(raw_cfg.keys()))

## 1.2 Board info

In [ ]:
print('Board:        ', soccfg['board'])
print('SW version:   ', soccfg['sw_version'])
print('FW timestamp: ', soccfg['fw_timestamp'])
print('Ref clk (MHz):', soccfg['refclk_freq'])

## 1.3 Signal generators

In [ ]:
print(f"Number of generators: {len(soccfg['gens'])}\n")
for i, gen in enumerate(soccfg['gens']):
    print(f'--- Gen ch {i} ---')
    for k, v in gen.items():
        print(f'  {k:20s}: {v}')

## 1.4 Readout channels

In [ ]:
print(f"Number of readouts: {len(soccfg['readouts'])}\n")
for i, ro in enumerate(soccfg['readouts']):
    print(f'--- Readout ch {i} ---')
    for k, v in ro.items():
        print(f'  {k:20s}: {v}')

## 1.5 tProcessor

In [ ]:
for i, tp in enumerate(soccfg['tprocs']):
    print(f'--- tProc {i} ---')
    for k, v in tp.items():
        print(f'  {k:20s}: {v}')

## 1.6 RF data converters (DACs / ADCs)

In [ ]:
rf = soccfg['rf']
print('RF keys:', list(rf.keys()))

print('\n--- DACs ---')
for name, dac in rf['dacs'].items():
    print(f'  DAC {name}: {dac}')

print('\n--- ADCs ---')
for name, adc in rf['adcs'].items():
    print(f'  ADC {name}: {adc}')

## 1.7 Optional blocks (IQ, DDR4, MR buffer, time taggers)

In [ ]:
for key in ['iqs', 'ddr4_buf', 'mr_buf', 'time_taggers']:
    val = raw_cfg.get(key)
    if val is not None:
        print(f'{key}: {val}')
    else:
        print(f'{key}: not present')

---
# 2. Frequency Conversion Methods

In [ ]:
gen_ch = 0
ro_ch = 0
test_freq = 5000.0  # MHz

# freq <-> register
reg = soccfg.freq2reg(test_freq, gen_ch=gen_ch)
freq_back = soccfg.reg2freq(reg, gen_ch=gen_ch)
print(f'freq2reg({test_freq} MHz, gen_ch={gen_ch}) = {reg}')
print(f'reg2freq({reg}, gen_ch={gen_ch})            = {freq_back:.6f} MHz')

# ADC frequency register
reg_adc = soccfg.freq2reg_adc(test_freq, ro_ch=ro_ch)
freq_adc_back = soccfg.reg2freq_adc(reg_adc, ro_ch=ro_ch)
print(f'\nfreq2reg_adc({test_freq} MHz, ro_ch={ro_ch}) = {reg_adc}')
print(f'reg2freq_adc({reg_adc}, ro_ch={ro_ch})          = {freq_adc_back:.6f} MHz')

# Round to valid frequency for both gen and readout
matched = soccfg.adcfreq(test_freq, gen_ch=gen_ch, ro_ch=ro_ch)
print(f'\nadcfreq({test_freq}, gen={gen_ch}, ro={ro_ch}) = {matched:.6f} MHz')

## 2.1 Frequency step sizes

In [ ]:
for i in range(len(soccfg['gens'])):
    gen_cfg = soccfg['gens'][i]
    step = soccfg.ch_fstep(gen_cfg)
    print(f'Gen ch {i}: freq step = {1e6*step:.6f} Hz')

print()
for i in range(len(soccfg['readouts'])):
    ro_cfg = soccfg['readouts'][i]
    step = soccfg.ch_fstep(ro_cfg)
    print(f'Readout ch {i}: freq step = {1e6*step:.6f} Hz')

---
# 3. Phase Conversion Methods

In [ ]:
test_deg = 90.0

for ch in range(min(len(soccfg['gens']), 4)):
    reg_phase = soccfg.deg2reg(test_deg, gen_ch=ch)
    deg_back = soccfg.reg2deg(reg_phase, gen_ch=ch)
    print(f'Gen ch {ch}: deg2reg({test_deg}) = {reg_phase},  reg2deg -> {deg_back:.4f}')

---
# 4. Timing Conversion Methods

In [ ]:
# tProc clock (default)
print('=== tProc / dispatcher clock ===')
print(f'1 cycle  = {soccfg.cycles2us(1):.6f} us')
print(f'1 us     = {soccfg.us2cycles(1)} cycles')

# Per-generator clocks
print('\n=== Generator fabric clocks ===')
for ch in range(min(len(soccfg['gens']), 4)):
    us_per_cycle = soccfg.cycles2us(1, gen_ch=ch)
    cycles_per_us = soccfg.us2cycles(1, gen_ch=ch)
    print(f'Gen ch {ch}: 1 cycle = {us_per_cycle:.6f} us,  1 us = {cycles_per_us} cycles')

# Per-readout clocks
print('\n=== Readout output clocks ===')
for ch in range(min(len(soccfg['readouts']), 4)):
    us_per_cycle = soccfg.cycles2us(1, ro_ch=ch)
    cycles_per_us = soccfg.us2cycles(1, ro_ch=ch)
    print(f'Readout ch {ch}: 1 cycle = {us_per_cycle:.6f} us,  1 us = {cycles_per_us} cycles')

---
# 5. Envelope Limits

In [ ]:
for ch in range(min(len(soccfg['gens']), 4)):
    print(f'Gen ch {ch}: max envelope amplitude = {soccfg.get_maxv(ch)}')

---
# 6. Per-channel detailed config

Key fields in each channel config dict: `f_fabric`, `f_output`, `f_dds`,
`b_dds`, `b_phase`, `has_mixer`, `interpolation`/`decimation`, etc.

In [ ]:
# Generator channel config via _get_ch_cfg
for ch in range(min(len(soccfg['gens']), 4)):
    ch_cfg = soccfg._get_ch_cfg(gen_ch=ch)
    print(f'--- Gen ch {ch} ---')
    print(f"  f_fabric  = {ch_cfg.get('f_fabric')} MHz")
    print(f"  f_output  = {ch_cfg.get('f_output')} MHz")
    print(f"  f_dds     = {ch_cfg.get('f_dds')} MHz")
    print(f"  b_dds     = {ch_cfg.get('b_dds')} bits")
    print(f"  b_phase   = {ch_cfg.get('b_phase')} bits")
    print(f"  has_mixer = {ch_cfg.get('has_mixer')}")
    print()

In [ ]:
# Readout channel config via _get_ch_cfg
for ch in range(min(len(soccfg['readouts']), 4)):
    ch_cfg = soccfg._get_ch_cfg(ro_ch=ch)
    print(f'--- Readout ch {ch} ---')
    print(f"  f_fabric  = {ch_cfg.get('f_fabric')} MHz")
    print(f"  f_output  = {ch_cfg.get('f_output')} MHz")
    print(f"  f_dds     = {ch_cfg.get('f_dds')} MHz")
    print(f"  b_dds     = {ch_cfg.get('b_dds')} bits")
    print(f"  b_phase   = {ch_cfg.get('b_phase')} bits")
    print()

---
# 7. `soc` (Pyro4 Proxy) Methods

The `soc` object is a remote proxy to the QICK hardware.
Below we query hardware-side methods.

In [ ]:
# List available remote methods
soc_methods = [m for m in dir(soc) if not m.startswith('_')]
print(f'Number of public methods/attrs: {len(soc_methods)}\n')
for m in sorted(soc_methods):
    print(f'  {m}')

## 7.1 RF board methods (if available)

These are present when the QICK board has an RF daughter board.

In [ ]:
rfb_methods = [m for m in dir(soc) if m.startswith('rfb_')]
print(f'RF board methods ({len(rfb_methods)}):')
for m in sorted(rfb_methods):
    print(f'  {m}')

## 7.2 Query RF board bias (if applicable)

In [ ]:
# Uncomment to read bias on a specific channel
# bias_ch = 0
# print(f'Bias ch {bias_ch}: {soc.rfb_get_bias(bias_ch)} V')

---
# 8. Sample Rates

Query DAC/ADC sampling rates from both `soccfg` (local config) and `soc` (remote hardware).

## 8.1 DAC sampling rates from `soccfg`

`fs` = actual DAC sampling rate (Msps), `f_fabric` = FPGA fabric clock,
`interpolation` = RF-DAC interpolation factor, `fs = f_ref * fs_mult / fs_div`.

In [ ]:
print('=== DAC Sampling Rates ===')
for name, dac in soccfg['rf']['dacs'].items():
    print(f"  DAC {name}: fs = {dac['fs']:.3f} Msps, "
          f"f_fabric = {dac['f_fabric']:.3f} MHz, "
          f"interpolation = {dac['interpolation']}x, "
          f"fs_mult = {dac['fs_mult']}, fs_div = {dac['fs_div']}, "
          f"f_ref = {dac['f_ref']:.2f} MHz")

## 8.2 ADC sampling rates from `soccfg`

`fs` = actual ADC sampling rate (Msps), `decimation` = RF-ADC decimation factor.

In [ ]:
print('=== ADC Sampling Rates ===')
for name, adc in soccfg['rf']['adcs'].items():
    print(f"  ADC {name}: fs = {adc['fs']:.3f} Msps, "
          f"f_fabric = {adc['f_fabric']:.3f} MHz, "
          f"decimation = {adc['decimation']}x, "
          f"fs_mult = {adc['fs_mult']}, fs_div = {adc['fs_div']}, "
          f"f_ref = {adc['f_ref']:.2f} MHz")

## 8.3 Generator effective sample rates

Each generator channel has a DAC behind it. `samps_per_clk` tells you how many
DAC samples are produced per fabric clock cycle.

In [ ]:
for i, gen in enumerate(soccfg['gens']):
    dacname = gen['dac']
    dac = soccfg['rf']['dacs'][dacname]
    print(f"Gen ch {i:2d}: DAC {dacname}, "
          f"fs = {dac['fs']:.3f} Msps, "
          f"f_fabric = {gen['f_fabric']:.3f} MHz, "
          f"samps_per_clk = {gen.get('samps_per_clk', 'N/A')}, "
          f"type = {gen['type']}")

## 8.4 Readout effective sample rates

Each readout channel has an ADC. `f_output` is the decimated output rate that
determines the time resolution of acquired data.

In [ ]:
for i, ro in enumerate(soccfg['readouts']):
    adcname = ro['adc']
    adc = soccfg['rf']['adcs'][adcname]
    print(f"Readout ch {i:2d}: ADC {adcname}, "
          f"fs = {adc['fs']:.3f} Msps, "
          f"f_output (decimated) = {ro['f_output']:.3f} MHz, "
          f"f_fabric = {ro['f_fabric']:.3f} MHz, "
          f"type = {ro['ro_type']}")

## 8.5 `soc.get_sample_rates()` — live hardware rates

This queries the actual RFDC tile sample rates from the running hardware.

In [ ]:
rates = soc.get_sample_rates()
print('Live sample rates from hardware:')
for k, v in rates.items():
    print(f'  {k}: {v}')

## 8.6 `soc.valid_sample_rates()` — allowed rates per tile

Shows what sample rates each DAC/ADC tile can be configured to.

In [ ]:
# Valid sample rates for each DAC tile
print('=== Valid DAC sample rates ===')
for tile in range(4):
    try:
        valid = soc.valid_sample_rates('dac', tile)
        print(f'  DAC tile {tile}: {valid}')
    except Exception as e:
        print(f'  DAC tile {tile}: {e}')

print('\n=== Valid ADC sample rates ===')
for tile in range(4):
    try:
        valid = soc.valid_sample_rates('adc', tile)
        print(f'  ADC tile {tile}: {valid}')
    except Exception as e:
        print(f'  ADC tile {tile}: {e}')

## 8.7 `soc.round_sample_rate()` — snap to nearest valid rate

Given a desired rate, returns the closest achievable sample rate.

In [ ]:
# Round a desired sample rate to nearest valid value
for target in [5000, 7000, 10000]:
    rounded = soc.round_sample_rate(target, 'dac', 0)
    print(f'  DAC tile 0: requested {target} Msps -> nearest valid = {rounded} Msps')

## 8.8 `soc.clocks_locked()` — check PLL lock status

In [ ]:
print('Clocks locked:', soc.clocks_locked())

---
# 9. Full raw config dump

Dump the entire `soccfg` dictionary for inspection.

In [ ]:
import json
print(json.dumps(soccfg.get_cfg(), indent=2, default=str))